In [1]:
import pandas as pd
from sqlalchemy import create_engine,URL,text

In [2]:
def data_extraction():
    try:
        connection_url = URL.create(
            drivername="mysql+pymysql",
            username="root",
            password="Shantanu@2026",
            host="localhost",
            port=3306,
            database="test_case_2"
        )

        engine = create_engine(connection_url)

        customer = pd.read_sql("SELECT * FROM customers", engine)
        orders = pd.read_sql("SELECT * FROM orders", engine)
        order_item = pd.read_sql("SELECT * FROM order_items", engine)
        product = pd.read_sql("SELECT * FROM products", engine)

        return customer, orders, order_item, product

    except Exception as e:
        print(f"Error: {e}")

In [3]:

def transformation(customer, order, order_item, product):

    # Remove duplicates
    customer.drop_duplicates(inplace=True)
    order.drop_duplicates(inplace=True)
    order_item.drop_duplicates(inplace=True)
    product.drop_duplicates(inplace=True)

    # Customer transformation
    customer = customer[customer["customer_name"].notna()]
    customer["city"] = customer["city"].fillna("Unknown")
    customer = customer[customer["email"].notna()]

    # Order transformation
    order = order[order["customer_id"].isin(customer["customer_id"])]
    order["order_date"] = order["order_date"].ffill()

    # Order item transformation
    order_item = order_item[order_item["order_id"].notna()]
    order_item = order_item[order_item["product_id"].notna()]

    order_item = order_item[
        order_item["order_id"].isin(order["order_id"])
    ]

    order_item.loc[order_item["quantity"] <= 0, "quantity"] = 1

    order_item = order_item[
        order_item["product_id"].isin(product["product_id"])
    ]

    # Product transformation
    product = product[product["product_id"].notna()]
    product["product_name"] = product["product_name"].fillna("Unknown")
    product["category"] = product["category"].fillna("Unknown")
    product = product[product["price"] > 0]

    return customer, order, order_item, product



In [4]:
def loading_data(customer, order, order_item, product):
        try:
            connection_url_2 = URL.create(
                drivername="mysql+pymysql",
                username="root",
                password="Shantanu@2026",
                host="localhost",
                port=3306,
                database="test_case_2_dw"
            )

            engine2 = create_engine(connection_url_2)\


            with engine2.begin() as conn:

                  # 1. Customers
                  customer.to_sql(
                  name="stagging_customers",
                  con=conn,
                  if_exists="append",
                  index=False
                  )

                  # 2. Products
                  product.to_sql(
                  name="stagging_products",
                  con=conn,
                  if_exists="append",
                  index=False
                  )

                  # 3. Orders
                  order.to_sql(
                  name="stagging_orders",
                  con=conn,
                  if_exists="append",
                  index=False
                  )

                  # 4. Order Items
                  order_item.to_sql(
                  name="stagging_order_items",
                  con=conn,
                  if_exists="append",
                  index=False
                  )

                  # Load dimension tables
                  conn.execute(text("""
                        INSERT INTO customer_dim
                              (customer_id, customer_name, city, email)
                        SELECT
                              customer_id,
                              customer_name,
                              city,
                              email
                        FROM stagging_customers
                  """))

                  conn.execute(text("""
                        INSERT INTO order_dim
                              (order_id, order_date)
                        SELECT
                              order_id,
                              order_date
                        FROM stagging_orders
                  """))

                  conn.execute(text("""
                  INSERT INTO product_dim
                        (product_id, product_name, category)
                  SELECT
                        product_id,
                        product_name,
                        category
                  FROM stagging_products
                  """))

                  conn.execute(text(
                        """INSERT INTO fact_total_sale (order_key,product_key,customer_key,price,quantity,total_sale)
                        SELECT o.order_key,p.product_key,c.customer_key,pr.price,soi.quantity,pr.price*soi.quantity as total_sale
                        FROM stagging_order_items soi
                        JOIN stagging_orders so
                              on so.order_id = soi.order_id 
                        JOIN order_dim o
                              on o.order_id=so.order_id
                        JOIN customer_dim c
                              on so.customer_id = c.customer_id
                        JOIN  product_dim p
                              on soi.product_id = p.product_id
                        JOIN stagging_products pr
                        on soi.product_id =pr.product_id;
                        """
                  ))

                  

        except Exception as e:
              print(f"Error : {e}")

    

In [ ]:
def main_function():
    try:
        customer, order, order_item, product=data_extraction()
        customer, order, order_item, product= transformation(customer, order, order_item, product)
    except Exception as e:
        print(f"Error : {e}")

    

In [6]:
main_function()

C:\Users\Cesar\AppData\Local\Temp\ipykernel_3800\1231636870.py:4: Pandas4Warning: The inplace keyword in DataFrame.drop_duplicates is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  customer.drop_duplicates(inplace=True)
C:\Users\Cesar\AppData\Local\Temp\ipykernel_3800\1231636870.py:5: Pandas4Warning: The inplace keyword in DataFrame.drop_duplicates is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  order.drop_duplicates(inplace=True)
C:\Users\Cesar\AppData\Local\Temp\ipykernel_3800\1231636870.py:6: Pandas4Warning: The inplace keyword in DataFrame.drop_duplicates is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  order_item.drop_duplicates(inplace=True)

Error : Execution failed on sql 'INSERT INTO stagging_customers (customer_id, customer_name, city, email) VALUES (:customer_id, :customer_name, :city, :email)': (pymysql.err.IntegrityError) (1062, "Duplicate entry '1' for key 'stagging_customers.PRIMARY'")
[SQL: INSERT INTO stagging_customers (customer_id, customer_name, city, email) VALUES (%(customer_id)s, %(customer_name)s, %(city)s, %(email)s)]
[parameters: [{'customer_id': 1, 'customer_name': 'Rahul', 'city': 'Pune', 'email': 'rahul@gmail.com'}, {'customer_id': 2, 'customer_name': 'Amit', 'city': 'Mumbai', 'email': 'amit@gmail.com'}, {'customer_id': 3, 'customer_name': 'Priya', 'city': 'Delhi', 'email': 'priya@gmail.com'}, {'customer_id': 4, 'customer_name': 'Sneha', 'city': 'Pune', 'email': 'sneha@gmail.com'}, {'customer_id': 5, 'customer_name': 'Rohan', 'city': 'Bangalore', 'email': 'rohan@gmail.com'}]]
(Background on this error at: https://sqlalche.me/e/21/gkpj)
